# NoCap-Test: Smart Filtering vs. baseline on Google Colab

Trains GPT-2 124M on FineWeb twice on the **same GPU**, once as the unmodified baseline and once with
Smart Filtering (Goldilocks zone + Early Ejection, see `IDEA.md`), then plots val loss against training time.

**Runtime → Change runtime type → GPU.** A100 or L4 can run the full benchmark; on a free T4 use the `quick` preset.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Setup
Logs and checkpoints go to Google Drive when it is mounted, so a disconnected session can be resumed.

In [ ]:
import os
USE_DRIVE = True  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/nocap'
else:
    OUT = '/content/nocap-out'
os.makedirs(OUT, exist_ok=True)

if not os.path.exists('/content/bottlecap-nocaptest'):
    !git clone https://github.com/vbtronic/bottlecap-nocaptest.git /content/bottlecap-nocaptest
%cd /content/bottlecap-nocaptest
!git pull -q
!pip install -q huggingface_hub

### 2. Configuration

| preset | tokens/step | steps | train tokens | ~time per run |
|---|---|---|---|---|
| `quick` | 131k | 1200 | 157M | T4 ≈ 1.5 h, A100 ≈ 15 min |
| `full` | 524k | 4768 (stops at val 3.3821) | 2.5B | A100 ≈ 3–4 h, T4 ≈ 20 h (too long for free Colab) |

`quick` does not reach 3.3821; compare the two curves at equal training time instead.

In [ ]:
PRESET = "quick"  #@param ["quick", "full"]
BATCH_SIZE = 16   #@param {type:"integer"}

if PRESET == "full":
    cfg = dict(tokens_per_step=524288, num_iterations=4768, warmup_iters=256, warmdown_iters=1024,
               val_loss_every=128, shards=33)
else:
    cfg = dict(tokens_per_step=131072, num_iterations=1200, warmup_iters=100, warmdown_iters=300,
               val_loss_every=100, shards=3)
SEQ = 1024
assert cfg['tokens_per_step'] % (BATCH_SIZE * SEQ) == 0
grad_accum = cfg['tokens_per_step'] // (BATCH_SIZE * SEQ)

def train_cmd(name, extra="", resume=""):
    cmd = (f"python train_gpt2.py --model d12 --batch_size {BATCH_SIZE} --grad_accumulation_steps {grad_accum} "
           f"--sequence_length {SEQ} --val_batch_size 16 --num_iterations {cfg['num_iterations']} "
           f"--learning_rate 0.0018 --weight_decay 0.1 --warmup_iters {cfg['warmup_iters']} "
           f"--warmdown_iters {cfg['warmdown_iters']} --val_loss_every {cfg['val_loss_every']} "
           f"--target_val_loss 3.3821 --save_every 200 "
           f"--output_dir {OUT}/{PRESET}/{name} --ckpt_dir {OUT}/{PRESET}/{name}/ckpt {extra}")
    if resume:
        cmd += f" --resume {resume}"
    return cmd + f" 2>&1 | tee -a {OUT}/{PRESET}/{name}.txt"

FILTER_ARGS = "--smart_filter --filter_pool 1.25 --filter_hard_frac 0.5 --filter_score_len 256 --filter_warmup 128"
os.makedirs(f"{OUT}/{PRESET}", exist_ok=True)
print(f"grad_accum={grad_accum}, shards={cfg['shards']}")

### 3. Data
Each shard is 100M tokens (~200 MB). Smart Filtering reads 25% more data than it trains on, hence the extra shards.

In [ ]:
!python data/cached_fineweb10B.py {cfg['shards']}
!du -sh data/fineweb10B

### 4. Baseline
To resume after a disconnect, set `RESUME` to the `latest.pt` path printed in the log (`.../ckpt/<run_id>/latest.pt`).

In [ ]:
RESUME = ""  #@param {type:"string"}
!{train_cmd("baseline", resume=RESUME)}

### 5. Smart Filtering

In [ ]:
RESUME = ""  #@param {type:"string"}
!{train_cmd("smartfilter", FILTER_ARGS, resume=RESUME)}

### 6. Compare

In [ ]:
import glob, re
import matplotlib.pyplot as plt

def read_val(name):
    logs = sorted(glob.glob(f"{OUT}/{PRESET}/{name}/*.log"), key=os.path.getmtime)
    if not logs:
        return [], [], []
    steps, vals, hours = [], [], []
    for line in open(logs[-1]):
        m = re.match(r"s:(\d+) val:([\d.]+) time:([\d.]+)", line)
        if m:
            steps.append(int(m[1])); vals.append(float(m[2])); hours.append(float(m[3]) / 3600)
    return steps, vals, hours

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ["baseline", "smartfilter"]:
    s, v, h = read_val(name)
    if not s:
        continue
    ax1.plot(h, v, marker=".", label=name)
    ax2.plot(s, v, marker=".", label=name)
    print(f"{name:12s} last: step {s[-1]}, val {v[-1]:.4f}, {h[-1]:.2f} h")
for ax, xl in [(ax1, "training time [h]"), (ax2, "step")]:
    ax.axhline(3.3821, color="gray", ls="--", lw=1)
    ax.set_xlabel(xl); ax.set_ylabel("val loss"); ax.set_ylim(top=6); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()